In [51]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os
import pickle

def preprocess_goldstein_uchida(file_path, save_output=True, output_dir='Other_Datasets/preprocessed_data'):
    """
    Preprocess/Noramlize the Goldstein-Uchida dataset.

    Args:
    file_path (str): Path to the input CSV file.
    save_output (bool): Whether to save the preprocessed data and metadata.
    output_dir (str): Directory to save the preprocessed data and metadata.

    Returns:
    pd.DataFrame: The normalized data.
    list: Indices of anomaly data points.
    dict: Mapping from original indices to preprocessed indices.
    """
    data = pd.read_csv(file_path, header=None)
    
    # remove the label column
    labels = data.iloc[:, -1]
    data = data.iloc[:, :-1]
    
    data = data.astype(float)
    anomaly_indices = labels[labels == 'o'].index.tolist()
    
    # Normalize numerical features
    scaler = StandardScaler()
    data = pd.DataFrame(scaler.fit_transform(data), columns=data.columns)
    normalized_data = range_based_normalize(data)
    
    original_to_preprocessed = dict(zip(data.index, normalized_data.index))
    
    if save_output:
        os.makedirs(output_dir, exist_ok=True)
        
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        
        normalized_data.to_csv(os.path.join(output_dir, f'preprocessed_{base_name}.csv'), index=False)
        np.save(os.path.join(output_dir, f'{base_name}_anomaly_indices.npy'), anomaly_indices)
        with open(os.path.join(output_dir, f'{base_name}_original_to_preprocessed.pkl'), 'wb') as f:
            pickle.dump(original_to_preprocessed, f)
        print(f"Preprocessed data, anomaly indices, and original to preprocessed index mapping saved to {output_dir}")
    
    return normalized_data, anomaly_indices, original_to_preprocessed

def range_based_normalize(data):
    """
    Normalize the data using range-based normalization.

    Args:
    data (pd.DataFrame): The input data to normalize.

    Returns:
    pd.DataFrame: The normalized data.
    """
    num_features = data.shape[1]
    max_value = 1 / num_features
    normalized_data = pd.DataFrame()

    for column in data.columns:
        min_val = data[column].min()
        max_val = data[column].max()
        normalized_data[column] = ((data[column] - min_val) / (max_val - min_val)) * max_value

    return normalized_data

In [52]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd

def perform_pca(data):
    """
    Perform PCA on the input data.
    
    Args:
    data (pd.DataFrame): Input data
    
    Returns:
    PCA: Fitted PCA object
    np.ndarray: Transformed data
    """
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data)
    
    pca = PCA()
    pca_data = pca.fit_transform(scaled_data)
    
    return pca, pca_data

def select_features(data, num_qubits, strategy='a'):
    """
    Select features based on the specified strategy.
    
    Args:
    data (pd.DataFrame): Input data
    num_qubits (int): Number of qubits specified in main
    strategy (str): Feature selection strategy (a, b, c, d, or e)
    
    Returns:
    pd.DataFrame: Data with selected features (including added 0-features if necessary)
    list: Indices of selected features
    """
    num_features = 2**num_qubits - 1
    original_num_features = data.shape[1]
    
    #if the number of features is greater than the original number of features, add zero features rather than selecting features
    if num_features >= original_num_features:
        selected_data = data.copy()
        num_zero_features = num_features - original_num_features
        for i in range(num_zero_features):
            selected_data[f'zero_feature_{i}'] = 0
        return selected_data, list(range(num_features))
    
    #uniform random selection of features
    if strategy == 'e':
        selected_features = np.random.choice(data.columns, num_features, replace=False)
        return data[selected_features], selected_features.tolist()
    
    pca, pca_data = perform_pca(data)
    feature_importance = np.abs(pca.components_).sum(axis=0)
    
    #Select the top num_features features
    if strategy == 'a':
        selected_indices = feature_importance.argsort()[::-1][:num_features]
    #Select the bottom num_features features
    elif strategy == 'b':
        selected_indices = feature_importance.argsort()[::-1][-num_features:]
    #Select the top half and bottom half of the features
    elif strategy == 'c':
        num_top = num_features // 2
        num_bottom = num_features - num_top
        top_indices = feature_importance.argsort()[::-1][:num_top]
        bottom_indices = feature_importance.argsort()[::-1][-num_bottom:]
        selected_indices = np.concatenate([top_indices, bottom_indices])
    
    #weighted random selection based on feature importance
    elif strategy == 'd':
        selected_indices = np.random.choice(
            len(feature_importance),
            num_features,
            replace=False,
            p=feature_importance / feature_importance.sum()
        )
    else:
        raise ValueError("Invalid strategy. Choose 'a', 'b', 'c', 'd', or 'e'.")
    
    selected_features = data.columns[selected_indices]
    return data[selected_features], selected_indices.tolist()

In [53]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, roc_auc_score, precision_recall_curve, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler
from keras import layers, losses
from keras.datasets import fashion_mnist
from keras.models import Model
from scipy.sparse import csr_matrix

import hashlib
import json
import ast

In [54]:
import pandas as pd
from sklearn.model_selection import train_test_split

# === 1. Preprocess the data ===
file_path = "Other_Datasets/letter-unsupervised-subset-ad.csv" 
normalized_data, anomaly_indices, mapping = preprocess_goldstein_uchida(file_path)

# Split normal vs anomaly
anomaly_data = normalized_data.iloc[anomaly_indices]
normal_data = normalized_data.drop(anomaly_indices)

# Train/test split (only normals in training)
X_train_normal, X_test_normal = train_test_split(
    normal_data,
    test_size=0.3,
    random_state=42
)

# Build train/test sets
X_train = X_train_normal
y_train = pd.Series([0]*len(X_train), index=X_train.index)

X_test = pd.concat([X_test_normal, anomaly_data])
y_test = pd.Series([0]*len(X_test_normal) + [1]*len(anomaly_data), index=X_test.index)

# === 2. Feature selection (8 features) ===
num_qubits = 3  # because 2^3 - 1 = 7, if you want 8 features you can just pad one zero
# OR set num_qubits = 4 → 2^4 - 1 = 15, then pick top 8 manually

# Here we force select_features to return 8 features
X_train_selected, selected_indices = select_features(X_train, num_qubits=4, strategy='c')
selected_indices = selected_indices[:8]   # keep only 8 features
X_train_selected = X_train.iloc[:, selected_indices]
X_test_selected = X_test.iloc[:, selected_indices]

print("Selected 8 feature indices:", selected_indices)


Preprocessed data, anomaly indices, and original to preprocessed index mapping saved to Other_Datasets/preprocessed_data
Selected 8 feature indices: [25, 21, 20, 24, 26, 12, 22, 16]


In [55]:
@tf.keras.utils.register_keras_serializable()
class AnomalyDetector(tf.keras.Model):
    def __init__(self, input_dim, **kwargs):
        super(AnomalyDetector, self).__init__(**kwargs)
        self.input_dim = input_dim
        self.encoder = tf.keras.Sequential([

            layers.Input(shape=(input_dim,)),
            layers.Dense(32, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Dropout(0.1),
            layers.Dense(16, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Dense(8, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
        ])
    
        self.decoder = tf.keras.Sequential([

            layers.Dense(16, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Dropout(0.1),
            layers.Dense(32, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Dense(input_dim) 

        ])
      
    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

    def get_config(self):
        # Return config dictionary; empty because no constructor args
        base_config = super(AnomalyDetector, self).get_config()
        base_config.update({"input_dim": self.input_dim})
        return base_config

    @classmethod
    def from_config(cls, config):
        return cls(**config)


autoencoder = AnomalyDetector(input_dim=8)

In [56]:
autoencoder.compile(optimizer='adam', loss='mae')

# Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

In [57]:
history = autoencoder.fit(X_train_selected, X_train_selected,
          epochs=20,
          batch_size=64, #Number of training samples model processes before updating its weights during training
          callbacks=[early_stopping, reduce_lr],
          shuffle=True)

Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0083 - learning_rate: 0.0010
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 982us/step - loss: 0.0054 - learning_rate: 0.0010
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 950us/step - loss: 0.0041 - learning_rate: 0.0010
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 970us/step - loss: 0.0038 - learning_rate: 0.0010
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0036 - learning_rate: 0.0010
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0034 - learning_rate: 0.0010
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 941us/step - loss: 0.0033 - learning_rate: 0.0010
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0034 - learning_rate: 0.0010
Epoch 9/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0034 - learning_rate: 0.0010
Epoch 10/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0035 - learning_rate: 0.0010
Epoch 11/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 936us/step - loss: 0.0034 - learning_rate: 0.0010


/home/r0han/miniconda3/envs/tf_env/lib/python3.10/site-packages/keras/src/callbacks/early_stopping.py:155: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss
  current = self.get_monitor_value(logs)
/home/r0han/miniconda3/envs/tf_env/lib/python3.10/site-packages/keras/src/callbacks/callback_list.py:151: UserWarning: Learning rate reduction is conditioned on metric `val_loss` which is not available. Available metrics are: loss,learning_rate.
  callback.on_epoch_end(epoch, logs)


Epoch 12/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 972us/step - loss: 0.0034 - learning_rate: 0.0010
Epoch 13/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0034 - learning_rate: 0.0010
Epoch 14/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 938us/step - loss: 0.0033 - learning_rate: 0.0010
Epoch 15/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 920us/step - loss: 0.0032 - learning_rate: 0.0010
Epoch 16/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 907us/step - loss: 0.0033 - learning_rate: 0.0010
Epoch 17/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 874us/step - loss: 0.0032 - learning_rate: 0.0010
Epoch 18/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 839us/step - loss: 0.0031 - learning_rate: 0.0010
Epoch 19/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 962us/step - loss: 0.0031 - learning_rate: 0.0010
Epoch 20/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 838us/step - loss: 0.0029 - learning_rate: 0.0010


In [50]:
autoencoder.save('Models/CAE/Others/Letter.keras')

In [58]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
import numpy as np

def evaluate_model(y_true, y_pred, scores, inference_time, threshold=0.2, model_name="Model"):
    """
    Evaluates a model and returns metrics formatted for copy-pasting.
    
    Parameters:
        y_true (np.array): Ground truth labels (0=normal, 1=anomaly).
        scores (np.array): Anomaly scores (lower = more anomalous).
        threshold (float): Threshold for anomaly decision.
        model_name (str): Name of the model.
        
    Returns:
        dict: Dictionary of metrics (can print or format for LaTeX).
    """

    precision = precision_score(y_true, y_pred, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auroc = roc_auc_score(y_true, scores)

    cm = confusion_matrix(y_true, y_pred)
    clf_report = classification_report(y_true, y_pred, digits=2)

    result = {
        "Model": model_name,
        "Precision": f"{precision:.3f}",
        "Accuracy": f"{accuracy:.3f}",
        "Recall": f"{recall:.3f}",
        "F1-Score": f"{f1:.3f}",
        "Matrix": f"TP={cm[1,1]}, FP={cm[0,1]}, FN={cm[1,0]}, TN={cm[0,0]}",
        "AUROC": f"{auroc:.3f}",
        "Inference Time (s)": f"{inference_time:.4f}",
        "Clf Report": clf_report.strip()
    }

    return result


In [59]:
import time

def predict(model, data, threshold):
    start_time = time.time()
    reconstructions = model(data)
    loss = tf.keras.losses.mae(reconstructions, data)
    # Detects if value greater than threshold then return true (for suspicious) else false (not suspicious)
    preds = tf.math.greater(loss, threshold)
    end_time = time.time()
    inference_time = end_time - start_time
    return loss.numpy(), preds.numpy().astype(int), inference_time

In [60]:
reconstructions = autoencoder.predict(X_train_selected)
reconstructions_error = tf.keras.losses.mae(reconstructions, X_train_selected)

# Calculate mean and std of the reconstruction errors
mean_error = np.mean(reconstructions_error)
std_error = np.std(reconstructions_error)

# Set threshold to be mean + N * std, where N is a hyperparameter you can tune
threshold = mean_error + 2 * std_error  # For a more aggressive threshold, decrease the multiplier

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


In [61]:
scores, y_pred, inference_time = predict(autoencoder, X_test_selected, threshold=threshold)

In [62]:
X_test = pd.concat([X_test_normal, anomaly_data])
y_test = pd.Series([0]*len(X_test_normal) + [1]*len(anomaly_data), index=X_test.index)

y_pred = y_pred = (scores > threshold).astype(int)
y_true = np.array(y_test).astype(int)[:len(scores)]

metrics = evaluate_model(
    y_true=y_true,
    y_pred=y_pred,
    scores=scores,
    inference_time=inference_time,
    threshold=threshold,
    model_name="8F Classical AE"
)

# Print clean table row for your notes
for k, v in metrics.items():
    print(f"{k}: {v}")

Model: 8F Classical AE
Precision: 0.636
Accuracy: 0.852
Recall: 0.424
F1-Score: 0.509
Matrix: TP=14, FP=8, FN=19, TN=142
AUROC: 0.880
Inference Time (s): 0.0083
Clf Report: precision    recall  f1-score   support

           0       0.88      0.95      0.91       150
           1       0.64      0.42      0.51        33

    accuracy                           0.85       183
   macro avg       0.76      0.69      0.71       183
weighted avg       0.84      0.85      0.84       183
